# CS+Social Good | Data Science Workshop 2: Machine Learning Deep Dive

**Today's question:** Can we predict whether a passenger survived the Titanic disaster?

We'll review Workshop 1, then learn **logistic regression**, **random forests**, and how to compare models fairly.

**Data:** [Kaggle Titanic competition](https://www.kaggle.com/competitions/titanic/data?select=train.csv). Download **`train.csv`** and put it beside this notebook. Kaggle's `test.csv` has no survival labels, so we will make our *own* train/test split from `train.csv`.

> This is a historical dataset for learning, not a model for real-world safety decisions.

## Quick review: The data science pipeline

**Question → Collect → Clean → Explore → Train → Evaluate → Improve**

| Step | In this workshop |
|---|---|
| Question | Can we predict survival? |
| Collect | Kaggle's passenger dataset |
| Clean | Fill missing ages and fares |
| Explore | Compare survival rates |
| Train | Build classification models |
| Evaluate | Check unseen passengers |
| Improve | Compare models and tune carefully |

**Remember:** We split before learning fill-in values, so the test set cannot give the model advance hints.

## Quick review: Workshop 1's models

| Model | Intuition | Example |
|---|---|---|
| **K-Nearest Neighbors** | Ask the most similar observations | "What happened to similar passengers?" |
| **Decision Tree** | Follow a sequence of yes/no questions | "First class? Older than 20?" |
| **Linear Regression** | Predict a **number** with a fitted relationship | "How much might a ticket cost?" |

**Important distinction:** Last time, linear regression predicted a *number*. Today, our target is **0 or 1** (did not survive / survived), so we need **classification** models.

```text
KNN:       nearby examples → majority vote
Tree:      question → question → prediction
Regression: input values → predicted number
```

#### **Optional review:** [StatQuest's video index](https://statquest.org/video_index.html) contains beginner explanations of these models:
- [StatQuest: K-nearest neighbors, Clearly Explained](https://www.youtube.com/watch?v=HVXime0nQeI)
- [Decision and Classification Trees, Clearly Explained!!!](https://www.youtube.com/watch?v=_L39rN6gz7Y)
- [Linear Regression, Clearly Explained!!!](https://www.youtube.com/watch?v=7ArmBVF2dCs)

## Setup

Install packages once (skip if already installed). In VS Code, select your Python kernel, then run cells with **Shift + Enter** (*or* click the "▶️" to the left of the cell).

In [ ]:
%pip install pandas matplotlib scikit-learn

## 1. Load and inspect the data

`Survived` is our **target**: **1 = survived**, **0 = did not survive**. We will use these features:

- `Pclass`: Ticket class (1st, 2nd, 3rd)
- `Sex`: Recorded sex
- `Age`: Passenger age
- `SibSp`: Siblings/spouses aboard
- `Parch`: Parents/children aboard
- `Fare`: Ticket fare
- `Embarked`: Port of departure

These historical categories are simplified, and relationships are **not** automatically causal.

In [ ]:
import pandas as pd

df = pd.read_csv("train.csv")
df.head()

In [ ]:
print("Rows, columns:", df.shape)
print("Missing values:")
print(df.isna().sum())

## 2. Explore one pattern

Which ticket classes had higher recorded survival rates? This is **EDA**, not proof that class itself caused survival.

In [ ]:
import matplotlib.pyplot as plt

df.groupby("Pclass")["Survived"].mean().plot(kind="bar")
plt.ylabel("Survival rate")
plt.xlabel("Ticket class")
plt.title("Survival rate by passenger class")
plt.ylim(0, 1)
plt.show()

## 3. Prepare our features

We **split** into training and testing passengers first. Then we:

1. Fill missing numeric values using training-set medians.
2. Fill missing text values using the training-set most common category.
3. Convert categories into 0/1 columns using **one-hot encoding**.

We only use features a passenger would have **before** knowing whether they survived.

In [ ]:
from sklearn.model_selection import train_test_split

columns = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[columns]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(len(X_train), "training passengers;", len(X_test), "test passengers")

In [ ]:
numeric = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
text = ["Sex", "Embarked"]

X_train = X_train.copy()
X_test = X_test.copy()

for col in numeric:
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

for col in text:
    common = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(common)
    X_test[col] = X_test[col].fillna(common)

X_train = pd.get_dummies(X_train, columns=text, dtype=int)
X_test = pd.get_dummies(X_test, columns=text, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_train.head()

## 4. New model: Logistic regression

**Despite its name, logistic regression is used for classification.** It first estimates a **probability** between 0 and 1, then applies a threshold (often 0.5) to pick a class.

```text
Passenger details → model → P(survived) = 0.72 → predict SURVIVED
Passenger details → model → P(survived) = 0.24 → predict DID NOT SURVIVE
```

Think of its probability curve as an **S-shape**: it stays between 0 and 1, unlike a straight line.

**Optional video:** [StatQuest: Logistic Regression](https://www.youtube.com/watch?v=yIYKR4sgzI8) (visual explanation).

We'll standardize numeric scales for this model. You do **not** need to memorize the scaling code today.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logistic.fit(X_train, y_train)

logistic_predictions = logistic.predict(X_test)
logistic_probabilities = logistic.predict_proba(X_test)[:, 1]

print("First five survival probabilities:", logistic_probabilities[:5].round(2))

### Visual: Probability vs. prediction

The model estimates a probability. **We** choose the threshold for converting it to a yes/no prediction.

**Question:** What changes if we require 70% confidence instead of 50%?

In [ ]:
plt.hist(logistic_probabilities, bins=15, edgecolor="white")
plt.axvline(0.5, color="red", linestyle="--", label="50% threshold")
plt.xlabel("Predicted probability of survival")
plt.ylabel("Passengers")
plt.title("Logistic regression: predicted probabilities")
plt.legend()
plt.show()

## 5. New model: Random Forest 🌲

A **single decision tree** makes one series of decisions. A **random forest** builds *many* slightly different trees and combines their votes.

```text
                        New passenger
                              |
              +---------------+---------------+
              |               |               |
           Tree 1          Tree 2          Tree 3    ...
           Survive         Survive         Not survive
              |               |               |
              +---------------+---------------+
                              |
                    Majority: Survive
```

Each tree sees a randomly sampled version of the training data and considers randomized subsets of features. That variety often makes the overall model more stable than one tree.

**Optional video:** [StatQuest: Random Forests, Part 1](https://www.youtube.com/watch?v=J4Wdy0Wc_xQ).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
forest.fit(X_train, y_train)

forest_predictions = forest.predict(X_test)

### Visual: What did the forest use?

**Feature importance** estimates which columns helped the forest make useful splits. It **does not** prove which factors caused survival, and importance values can be misleading with correlated features.

In [ ]:
importance = pd.Series(forest.feature_importances_, index=X_train.columns)
importance.sort_values().plot(kind="barh")
plt.xlabel("Feature importance (relative)")
plt.title("What did the random forest use?")
plt.show()

## 6. Compare our models

**Accuracy** = share of test predictions that were correct. Let's also include our Workshop 1 decision tree as a baseline.

We compare all models on **exactly the same test passengers**.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
tree_predictions = tree.predict(X_test)

results = pd.Series({
    "Decision Tree": accuracy_score(y_test, tree_predictions),
    "Logistic Regression": accuracy_score(y_test, logistic_predictions),
    "Random Forest": accuracy_score(y_test, forest_predictions)
}).sort_values(ascending=False)

print(results.round(3))
results.plot(kind="bar", ylim=(0, 1), ylabel="Test accuracy", title="Model comparison")
plt.xticks(rotation=20, ha="right")
plt.show()

## 7. Beyond accuracy: Confusion matrix

Accuracy doesn't tell us *which mistakes* the model made.

| | **Predicted no** | **Predicted yes** |
|---|---|---|
| **Actually no** | True negative | False positive |
| **Actually yes** | False negative | True positive |

- **Precision:** Of the passengers predicted to survive, what fraction actually did?
- **Recall:** Of the passengers who survived, what fraction did we correctly identify?

**Optional video:** [StatQuest: Confusion Matrix](https://www.youtube.com/watch?v=Kdsp6soqA7o).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

ConfusionMatrixDisplay.from_predictions(
    y_test, forest_predictions, display_labels=["Did not survive", "Survived"],
    cmap="Blues", colorbar=False
)
plt.title("Random forest confusion matrix")
plt.show()
print(classification_report(y_test, forest_predictions, target_names=["Did not survive", "Survived"]))

## 8. Extra concept: Overfitting

Imagine memorizing answers to a practice quiz rather than understanding the concepts. You might ace the practice questions and struggle on new ones.

- **Overfitting:** Excellent on training data, worse on unseen test data.
- **Underfitting:** Too simple to capture patterns, even in training data.

Let's compare a very deep tree with a smaller one.

In [ ]:
deep_tree = DecisionTreeClassifier(random_state=42)
deep_tree.fit(X_train, y_train)

print("Deep tree training accuracy:", round(deep_tree.score(X_train, y_train), 3))
print("Deep tree test accuracy:    ", round(deep_tree.score(X_test, y_test), 3))
print("Small tree training accuracy:", round(tree.score(X_train, y_train), 3))
print("Small tree test accuracy:    ", round(tree.score(X_test, y_test), 3))

## 9. Optional extension: Cross-validation

A single train/test split can be lucky or unlucky. **Cross-validation** repeats training and evaluation using different folds of the *training data*.

This example scores a random forest over 5 folds. We keep the final test data untouched.

**Optional video:** [StatQuest: Cross Validation](https://www.youtube.com/watch?v=fSytzGwwBVw).

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(forest, X_train, y_train, cv=5, scoring="accuracy")
print("Fold scores:", scores.round(3))
print("Average CV accuracy:", round(scores.mean(), 3))

## 10. Try it yourself

1. Set `n_estimators=10` in the forest. Does accuracy change? Try `200`.
2. Change `max_depth=5` to `2`, then `None`. What changes?
3. Change the logistic classification threshold from **0.5** to **0.7**. Does the number predicted to survive increase or decrease?
4. Which model would you use here, and why? Is accuracy the only consideration?

**Challenge (optional):** Inspect whether survival rates differ by `Sex`, `Age`, or `Embarked`. What can your visualizations show, and what *can't* they establish?

### Key takeaways

- Logistic regression predicts **probabilities** for classification.
- Random forests combine **many decision trees**.
- Compare models on the **same held-out test set**.
- Confusion matrices reveal mistakes accuracy can hide.
- A model should **generalize**, not just memorize.

**Remember:** Historical correlations do not establish causation, and models can learn patterns shaped by social conditions and unequal treatment.

In [ ]:
threshold = 0.7
new_predictions = (logistic_probabilities >= threshold).astype(int)
print("Predicted survivors at 50%:", logistic_predictions.sum())
print("Predicted survivors at 70%:", new_predictions.sum())